# Spark + Lance internal mode

Register a Lance INTERNAL catalog in Kasanari and run a Spark SQL lifecycle against it.

In [1]:
import json
import requests

base_url = "http://kasanari:9090"
catalog_id = "lance_spark_internal"

payload = {
    "catalogId": catalog_id,
    "catalogType": "LANCE",
    "mode": "INTERNAL",
    "spec": {
        "fileIoProperties": {},
        "catalogProperties": {
            "implementation": "kasanari.catalog.lance.KasanariLanceCatalog",
            "kasanari.jdbc.user": "postgres",
            "kasanari.jdbc.password": "postgres",
            "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
            "lance.warehouse.location": "s3://warehouse",
            "lance.storage.aws_access_key_id": "admin",
            "lance.storage.aws_secret_access_key": "password",
            "lance.storage.aws_endpoint": "http://minio:9000",
            "lance.storage.aws_allow_http": "true",
            "lance.storage.aws_virtual_hosted_style_request": "false",
            "lance.storage.region": "us-east-1"
        }
    }
}

response = requests.post(f"{base_url}/management/v1/catalogs", json=payload, timeout=20)
print(response.status_code)
print(response.text)


409
{"message":"Catalog already exists"}


In [2]:
response = requests.get(f"{base_url}/management/v1/catalogs/LANCE/{catalog_id}", timeout=20)
print(response.status_code)
print(json.dumps(response.json(), indent=2))


200
{
  "catalogId": "lance_spark_internal",
  "catalogType": "LANCE",
  "mode": "INTERNAL",
  "spec": {
    "fileIoProperties": {},
    "catalogProperties": {
      "implementation": "kasanari.catalog.lance.KasanariLanceCatalog",
      "kasanari.jdbc.user": "postgres",
      "kasanari.jdbc.password": "postgres",
      "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
      "lance.warehouse.location": "s3://warehouse",
      "lance.storage.aws_access_key_id": "admin",
      "lance.storage.aws_secret_access_key": "password",
      "lance.storage.aws_endpoint": "http://minio:9000",
      "lance.storage.aws_allow_http": "true",
      "lance.storage.aws_virtual_hosted_style_request": "false",
      "lance.storage.region": "us-east-1"
    }
  },
  "version": 1
}


## Spark SQL operations through Lance REST catalog

This section initializes the Lance Spark connector and executes create/insert/select/update/alter/view/delete/drop operations.

In [5]:
import uuid
from pyspark.sql import SparkSession

spark_catalog = "kasanari_lance"

spark = (
    SparkSession.builder
    .appName("kasanari-lance-internal-ops")
    .master("local[*]")
    .config("spark.jars", "/home/jovyan/extra-jars/lance-spark-runtime-4_2.13-0.4.0.jar")
    .config("spark.sql.extensions", "org.lance.spark.extensions.LanceSparkSessionExtensions")
    .config(f"spark.sql.catalog.{spark_catalog}", "org.lance.spark.LanceNamespaceSparkCatalog")
    .config(f"spark.sql.catalog.{spark_catalog}.impl", "rest")
    .config(f"spark.sql.catalog.{spark_catalog}.uri", f"{base_url}/lance")
    .config(f"spark.sql.catalog.{spark_catalog}.parent", catalog_id)
    .config(f"spark.sql.catalog.{spark_catalog}.parent_delimiter", ".")
    .config(f"spark.sql.catalog.{spark_catalog}.storage.access_key_id", "admin")
    .config(f"spark.sql.catalog.{spark_catalog}.storage.secret_access_key", "password")
    .config(f"spark.sql.catalog.{spark_catalog}.storage.endpoint", "http://minio:9000")
    .config(f"spark.sql.catalog.{spark_catalog}.storage.aws_allow_http", "true")
    .config("spark.sql.defaultCatalog", spark_catalog)
    .getOrCreate()
)

print("Connector creation:")
print(f"Configured Spark catalog {spark_catalog} using Lance REST namespace")

namespace = "demo"
table = f"events_{uuid.uuid4().hex[:8]}"
view = f"{table}_v"
table_name = f"{spark_catalog}.{namespace}.{table}"
view_name = f"{spark_catalog}.{namespace}.{view}"

def run_sql(label, statement, show=False, optional=False):
    print(f"\n== {label} ==")
    print(statement.strip())
    try:
        result = spark.sql(statement)
        if show:
            result.show(truncate=False)
        else:
            print("OK")
        return True
    except Exception as exc:
        status = "SKIPPED (not supported in this environment)" if optional else "FAILED"
        print(f"{status}: {exc}")
        return False

run_sql("Namespace create", f"CREATE NAMESPACE IF NOT EXISTS {spark_catalog}.{namespace}")

run_sql(
    "Table create",
    f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
      id INT,
      event_name STRING,
      amount DOUBLE,
      source STRING
    )
    """
)

run_sql(
    "Insert into table",
    f"""
    INSERT INTO {table_name} (id, event_name, amount, source)
    VALUES
      (1, 'signup', 10.0, 'seed'),
      (2, 'click', 20.0, 'seed'),
      (3, 'purchase', 30.0, 'seed')
    """
)

run_sql(
    "Select",
    f"SELECT id, event_name, amount, source FROM {table_name} ORDER BY id",
    show=True
)

run_sql(
    "Update",
    f"UPDATE {table_name} SET source = 'updated' WHERE id IN (1, 2)"
)

run_sql(
    "Select",
    f"SELECT id, event_name, amount, source FROM {table_name} ORDER BY id",
    show=True
)

run_sql(
    "Alter",
    f"ALTER TABLE {table_name} ADD COLUMN notes STRING"
)

run_sql(
    "Select",
    f"SELECT id, event_name, amount, source, notes FROM {table_name} ORDER BY id",
    show=True
)

view_created = run_sql(
    "Create view",
    f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT id, event_name, amount
    FROM {table_name}
    WHERE id <= 2
    """,
    optional=True
)

if view_created:
    run_sql(
        "Select from view",
        f"SELECT id, event_name, amount FROM {view_name} ORDER BY id",
        show=True,
        optional=True
    )
else:
    print("\n== Select from view ==")
    print("SKIPPED (view was not created)")

run_sql(
    "Delete from table",
    f"DELETE FROM {table_name} WHERE id = 3"
)

run_sql(
    "Select",
    f"SELECT id, event_name, amount, source, notes FROM {table_name} ORDER BY id",
    show=True
)

table_dropped = run_sql("Drop table", f"DROP TABLE IF EXISTS {table_name}", optional=True)
run_sql("Drop view", f"DROP VIEW IF EXISTS {view_name}", optional=True)

if not table_dropped:
    print("\nRetry cleanup using view-first order")
    run_sql("Fallback drop view", f"DROP VIEW IF EXISTS {view_name}", optional=True)
    run_sql("Fallback drop table", f"DROP TABLE IF EXISTS {table_name}", optional=True)

print("\nDone: completed Lance Spark lifecycle sample.")


Connector creation:
Configured Spark catalog kasanari_lance using Lance REST namespace

== Namespace create ==
CREATE NAMESPACE IF NOT EXISTS kasanari_lance.demo
OK

== Table create ==
CREATE TABLE IF NOT EXISTS kasanari_lance.demo.events_6308584c (
      id INT,
      event_name STRING,
      amount DOUBLE,
      source STRING
    )
OK

== Insert into table ==
INSERT INTO kasanari_lance.demo.events_6308584c (id, event_name, amount, source)
    VALUES
      (1, 'signup', 10.0, 'seed'),
      (2, 'click', 20.0, 'seed'),
      (3, 'purchase', 30.0, 'seed')
OK

== Select ==
SELECT id, event_name, amount, source FROM kasanari_lance.demo.events_6308584c ORDER BY id
+---+----------+------+------+
|id |event_name|amount|source|
+---+----------+------+------+
|1  |signup    |10.0  |seed  |
|2  |click     |20.0  |seed  |
|3  |purchase  |30.0  |seed  |
+---+----------+------+------+


== Update ==
UPDATE kasanari_lance.demo.events_6308584c SET source = 'updated' WHERE id IN (1, 2)
OK

== Select 

{"ts": "2026-06-10 02:10:00.128", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o53.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703; line 1 pos 39;\n'Sort ['id ASC NULLS FIRST], true\n+- 'Project [id#168, event_name#169, amount#170, source#171, 'notes]\n   +- SubqueryAlias kasanari_lance.demo.events_6308584c\n      +- RelationV2[id#168, event_name#169, amount#170, source#171] kasanari_lance.demo.events_6308584c kasana

FAILED: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703; line 1 pos 39;
'Sort ['id ASC NULLS FIRST], true
+- 'Project [id#168, event_name#169, amount#170, source#171, 'notes]
   +- SubqueryAlias kasanari_lance.demo.events_6308584c
      +- RelationV2[id#168, event_name#169, amount#170, source#171] kasanari_lance.demo.events_6308584c kasanari_lance.demo.events_6308584c


== Create view ==
CREATE OR REPLACE VIEW kasanari_lance.demo.events_6308584c_v AS
    SELECT id, event_name, amount
    FROM kasanari_lance.demo.events_6308584c
    WHERE id <= 2
SKIPPED (not supported in this environment): Catalog kasanari_lance does not support views.

== Select from view ==
SKIPPED (view was not created)

== Delete from table ==
DELETE FROM kasanari_lance.demo.events_6308584c WHERE id = 3


{"ts": "2026-06-10 02:10:01.629", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o53.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703; line 1 pos 39;\n'Sort ['id ASC NULLS FIRST], true\n+- 'Project [id#205, event_name#206, amount#207, source#208, 'notes]\n   +- SubqueryAlias kasanari_lance.demo.events_6308584c\n      +- RelationV2[id#205, event_name#206, amount#207, source#208] kasanari_lance.demo.events_6308584c kasana

OK

== Select ==
SELECT id, event_name, amount, source, notes FROM kasanari_lance.demo.events_6308584c ORDER BY id
FAILED: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the following? [`id`, `source`, `amount`, `event_name`]. SQLSTATE: 42703; line 1 pos 39;
'Sort ['id ASC NULLS FIRST], true
+- 'Project [id#205, event_name#206, amount#207, source#208, 'notes]
   +- SubqueryAlias kasanari_lance.demo.events_6308584c
      +- RelationV2[id#205, event_name#206, amount#207, source#208] kasanari_lance.demo.events_6308584c kasanari_lance.demo.events_6308584c


== Drop table ==
DROP TABLE IF EXISTS kasanari_lance.demo.events_6308584c
OK

== Drop view ==
DROP VIEW IF EXISTS kasanari_lance.demo.events_6308584c_v
SKIPPED (not supported in this environment): [UNSUPPORTED_FEATURE.CATALOG_OPERATION] The feature is not supported: Catalog `kasanari_lance` does not support views. SQLSTATE: 0A000

Done: completed La